# 4. Ingeniería de atributos

**Issue:** [#4 Feature Engineering](https://github.com/velascocafe23/telco-churn-mlops/issues/4)

Este notebook construye el pipeline de preprocesamiento que transforma los datos
intermedios en atributos aptos para el entrenamiento.

Es la pieza estructural del proyecto. El objeto que se produce aquí se reutiliza en el
modelo base, en la selección del mejor modelo, se serializa junto al modelo definitivo, y
lo cargan después el pipeline de inferencia y la aplicación de despliegue. Una
inconsistencia en esta etapa se propaga a todas las demás, y un preprocesamiento que se
aplique distinto en entrenamiento y en producción es la causa más frecuente de que un
modelo funcione bien en el notebook y mal en operación.

## 4.1 Configuración

In [1]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import KBinsDiscretizer, OneHotEncoder, StandardScaler

INTERMEDIATE_FILE = Path("../../data/02_intermediate/telco_customer_churn.parquet")
MODELS_DIR = Path("../../models")
PREPROCESADOR_FILE = MODELS_DIR / "preprocesador.joblib"

IDENTIFICADOR = "customerID"
OBJETIVO = "Churn"
CLASE_POSITIVA = "Yes"

COLUMNAS_NUMERICAS = ["tenure", "MonthlyCharges", "TotalCharges"]
COLUMNA_ANTIGUEDAD = ["tenure"]

N_TRAMOS = 5
PROPORCION_PRUEBA = 0.2
SEMILLA = 42

set_config(display="diagram", transform_output="pandas")

datos = pd.read_parquet(INTERMEDIATE_FILE)

COLUMNAS_CATEGORICAS = [
    columna for columna in datos.select_dtypes(include="category").columns if columna != OBJETIVO
]

print(f"Registros:   {len(datos)}")
print(f"Numéricas:   {COLUMNAS_NUMERICAS}")
print(f"Categóricas: {len(COLUMNAS_CATEGORICAS)} atributos")

Registros:   7043
Numéricas:   ['tenure', 'MonthlyCharges', 'TotalCharges']
Categóricas: 16 atributos


## 4.2 Decisiones de preprocesamiento

Cada decisión proviene de una evidencia concreta del análisis exploratorio, no de una
receta genérica.

| Decisión | Evidencia que la sustenta |
|---|---|
| Descartar `customerID` | Identificador único, sin valor predictivo |
| Imputar `TotalCharges` con cero | Los once ausentes son estructurales: clientes con antigüedad cero, sin primera factura emitida |
| Discretizar `tenure` en tramos | Distribución bimodal y fuerte interacción con el tipo de contrato |
| Conservar `tenure` también como continua | Los modelos de árboles aprovechan el orden; la discretización se agrega, no reemplaza |
| Conservar categorías `No internet service` | Codifican la ausencia del servicio base, información distinta de no contratar el complemento |
| Codificación por indicadores | Cardinalidad baja, máximo cuatro categorías por atributo |
| Estandarizar las numéricas | Necesario para modelos sensibles a la escala, inocuo para árboles |
| No eliminar registros duplicados | Son clientes distintos con perfil coincidente; eliminarlos sesgaría la muestra |
| No tratar valores atípicos | No hay atípicos por el criterio del rango intercuartílico |

## 4.3 Separación de atributos y objetivo

In [2]:
atributos = datos.drop(columns=[IDENTIFICADOR, OBJETIVO])
objetivo = (datos[OBJETIVO] == CLASE_POSITIVA).astype(int)

print(f"Atributos: {atributos.shape}")
print(f"Objetivo:  {objetivo.shape}, tasa positiva {objetivo.mean():.4f}")
print()
print(f"Columnas de entrada: {atributos.columns.tolist()}")

Atributos: (7043, 19)
Objetivo:  (7043,), tasa positiva 0.2654

Columnas de entrada: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


## 4.4 Construcción del pipeline

El preprocesador se arma con tres ramas sobre un `ColumnTransformer`:

1. **Numérica**: imputación con cero y estandarización.
2. **Antigüedad discretizada**: la misma columna `tenure` entra por segunda vez, ahora
   para producir tramos codificados como indicadores.
3. **Categórica**: codificación por indicadores con tolerancia a categorías no vistas.

Que una columna alimente dos ramas es intencional: el modelo recibe la antigüedad como
magnitud continua y como pertenencia a un tramo, y decide cuál le sirve.

In [3]:
rama_numerica = Pipeline(
    steps=[
        ("imputacion", SimpleImputer(strategy="constant", fill_value=0)),
        ("escalado", StandardScaler()),
    ]
)

rama_antiguedad = Pipeline(
    steps=[
        (
            "tramos",
            KBinsDiscretizer(
                n_bins=N_TRAMOS,
                encode="onehot-dense",
                strategy="uniform",
            ),
        ),
    ]
)

rama_categorica = Pipeline(
    steps=[
        (
            "indicadores",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        ),
    ]
)

preprocesador = ColumnTransformer(
    transformers=[
        ("numericas", rama_numerica, COLUMNAS_NUMERICAS),
        ("antiguedad", rama_antiguedad, COLUMNA_ANTIGUEDAD),
        ("categoricas", rama_categorica, COLUMNAS_CATEGORICAS),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

preprocesador

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numericas', ...), ('antiguedad', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` an

## 4.5 Verificacion sobre una particion

El ajuste se hace **solo sobre el conjunto de entrenamiento**. Ajustar el escalador sobre
el conjunto completo filtraria al entrenamiento la media y la desviacion de los datos de
prueba, que es una forma sutil pero real de fuga de informacion.

Se reinicia ademas el indice de ambas particiones. Con la salida de los transformadores
configurada como DataFrame, cada rama del `ColumnTransformer` devuelve su resultado con
un indice propio, y si el indice de entrada viene disperso, la concatenacion de las ramas
produce un desalineamiento. Es un detalle de implementacion, pero del tipo que aparece en
produccion y no en el notebook, porque depende de como venga indexado el lote de entrada.


In [4]:
atributos_entrenamiento, atributos_prueba, objetivo_entrenamiento, objetivo_prueba = (
    train_test_split(
        atributos,
        objetivo,
        test_size=PROPORCION_PRUEBA,
        random_state=SEMILLA,
        stratify=objetivo,
    )
)

# La particion hereda el indice original, que queda disperso. Se reinicia para que las
# ramas del ColumnTransformer produzcan salidas alineadas al concatenarlas.
atributos_entrenamiento = atributos_entrenamiento.reset_index(drop=True)
atributos_prueba = atributos_prueba.reset_index(drop=True)
objetivo_entrenamiento = objetivo_entrenamiento.reset_index(drop=True)
objetivo_prueba = objetivo_prueba.reset_index(drop=True)

print(f"Entrenamiento: {atributos_entrenamiento.shape}")
print(f"Prueba:        {atributos_prueba.shape}")
print(f"Tasa positiva entrenamiento: {objetivo_entrenamiento.mean():.4f}")
print(f"Tasa positiva prueba:        {objetivo_prueba.mean():.4f}")

Entrenamiento: (5634, 19)
Prueba:        (1409, 19)
Tasa positiva entrenamiento: 0.2654
Tasa positiva prueba:        0.2654


In [5]:
transformados_entrenamiento = preprocesador.fit_transform(atributos_entrenamiento)
transformados_prueba = preprocesador.transform(atributos_prueba)

print(f"Entrada:  {atributos_entrenamiento.shape[1]} columnas")
print(f"Salida:   {transformados_entrenamiento.shape[1]} columnas")
print(f"Prueba:   {transformados_prueba.shape}")

Entrada:  19 columnas
Salida:   51 columnas
Prueba:   (1409, 51)


In [6]:
nombres_salida = preprocesador.get_feature_names_out()

origen = pd.Series(nombres_salida).str.split("__").str[0].value_counts()
print("Atributos generados por rama")
print(origen.to_string())
print()
print("Primeros quince nombres")
for nombre in nombres_salida[:15]:
    print(f"  {nombre}")

Atributos generados por rama
categoricas    43
antiguedad      5
numericas       3

Primeros quince nombres
  numericas__tenure
  numericas__MonthlyCharges
  numericas__TotalCharges
  antiguedad__tenure_0.0
  antiguedad__tenure_1.0
  antiguedad__tenure_2.0
  antiguedad__tenure_3.0
  antiguedad__tenure_4.0
  categoricas__gender_Female
  categoricas__gender_Male
  categoricas__SeniorCitizen_No
  categoricas__SeniorCitizen_Yes
  categoricas__Partner_No
  categoricas__Partner_Yes
  categoricas__Dependents_No


In [7]:
columnas_escaladas = [f"numericas__{columna}" for columna in COLUMNAS_NUMERICAS]

diagnostico = pd.DataFrame(
    {
        "media": transformados_entrenamiento[columnas_escaladas].mean(),
        "desviacion": transformados_entrenamiento[columnas_escaladas].std(),
        "minimo": transformados_entrenamiento[columnas_escaladas].min(),
        "maximo": transformados_entrenamiento[columnas_escaladas].max(),
    }
).round(4)

print(
    f"Valores ausentes tras la transformación: {int(transformados_entrenamiento.isna().sum().sum())}"
)
print()
diagnostico

Valores ausentes tras la transformación: 0



,media,desviacion,minimo,maximo
numericas__tenure,-0.0,1.0001,-1.3223,1.6085
numericas__MonthlyCharges,-0.0,1.0001,-1.5440,1.7859
numericas__TotalCharges,0.0,1.0001,-1.0089,2.8019


Las numéricas quedan centradas y con desviación unitaria, no hay ausentes en la salida, y
el conjunto de prueba se transforma con los parámetros aprendidos en entrenamiento.

Vale la pena notar que el mínimo de `TotalCharges` escalado corresponde a los clientes
imputados con cero. Quedan en el extremo inferior de la distribución, que es exactamente
donde deben estar: son los clientes sin historia de facturación.

## 4.6 Nota sobre el atributo derivado y la serialización

El análisis exploratorio propuso un atributo derivado: el número de servicios contratados
por cliente, como medida de vinculación con la compañía. **No se incluye en este
pipeline**, y la razón es de despliegue, no de modelado.

Calcular ese atributo exige una función propia dentro del pipeline. Al serializar, la
función no se guarda: lo que queda almacenado es una referencia al módulo donde estaba
definida. Si esa función vive en el espacio de nombres de un notebook, el módulo de
referencia es el intérprete interactivo, y al cargar el objeto desde otro proceso, como
el script de inferencia o la aplicación web, la referencia no resuelve y la carga falla.

El pipeline construido aquí usa exclusivamente transformadores de la librería, de modo
que se serializa y se recupera desde cualquier proceso sin dependencias del entorno donde
fue creado.

El atributo derivado se incorpora en la etapa de arquitectura FTI, donde el código vive
en un módulo importable bajo `src/` y la referencia sí resuelve correctamente. Esta
restricción es un buen ejemplo de por qué el paso de notebooks a scripts no es un trámite
de ordenamiento: habilita cosas que en el notebook no son viables.

## 4.7 Persistencia del preprocesador

Se guarda el pipeline **sin ajustar**. Guardar la versión ajustada sobre estos datos
fijaría las estadísticas de escalado y las categorías aprendidas de esta partición
concreta, y cada etapa posterior debe ajustarlo sobre su propio conjunto de
entrenamiento, dentro de la validación cruzada que corresponda.

Lo que se comparte entre etapas es **la definición de las transformaciones**, no los
parámetros aprendidos. El objeto ajustado que sí se serializa es el del modelo final, en
el issue de selección de modelo, donde preprocesador y estimador viajan juntos.

In [8]:
preprocesador_sin_ajustar = ColumnTransformer(
    transformers=[
        ("numericas", rama_numerica, COLUMNAS_NUMERICAS),
        ("antiguedad", rama_antiguedad, COLUMNA_ANTIGUEDAD),
        ("categoricas", rama_categorica, COLUMNAS_CATEGORICAS),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(preprocesador_sin_ajustar, PREPROCESADOR_FILE)

print(f"Guardado en: {PREPROCESADOR_FILE.resolve()}")
print(f"Tamaño: {PREPROCESADOR_FILE.stat().st_size / 1024:.1f} KB")

Guardado en: /home/velas/proyectos/telco-churn-mlops/models/preprocesador.joblib
Tamaño: 1.4 KB


In [9]:
recuperado = joblib.load(PREPROCESADOR_FILE)
prueba_recuperado = recuperado.fit_transform(atributos_entrenamiento)

assert prueba_recuperado.shape == transformados_entrenamiento.shape, "Dimensiones distintas"
assert list(recuperado.get_feature_names_out()) == list(nombres_salida), "Atributos distintos"

print("El preprocesador recuperado reproduce la misma transformación")
print(f"Atributos de salida: {prueba_recuperado.shape[1]}")

El preprocesador recuperado reproduce la misma transformación
Atributos de salida: 51


## 4.8 Conclusiones

1. El preprocesamiento queda encapsulado en un único objeto que aplica de forma idéntica
   imputación, discretización, escalado y codificación.
2. Cada decisión de transformación está sustentada en evidencia del análisis exploratorio.
3. El ajuste se realiza exclusivamente sobre el conjunto de entrenamiento, lo que evita
   la fuga de información a través de las estadísticas de escalado.
4. La codificación tolera categorías no vistas, condición necesaria para que la
   inferencia no falle ante un valor nuevo en producción.
5. El pipeline no depende de código definido en el notebook, de modo que puede cargarse
   desde cualquier proceso.
6. Se persiste la definición sin ajustar, para que cada etapa posterior lo ajuste sobre
   su propia partición.

**Siguiente paso:** modelo base de referencia, usando este pipeline.